In [5]:
import example_loader as el
import gurobipy as gp
import gurobi_utils as gu
import miplib_loader as ml
import numpy as np
import plot_utils as pu

In [6]:
# configure the backend for matplotlib
# this one should allow zoom:
%matplotlib widget
# to make that work you need: "pip install ipympl" and run "jupyter nbextension enable --py widgetsnbextension"

# this will work without the above dependencies but won't allow zoom
# %matplotlib inline

# this option may work whenever they fix bugs in mpld3
# import mpld3
# mpld3.enable_notebook()

In [ ]:
# A function to create cuts given a target point
def add_balas_ball_cut(relaxed: gp.Model, target, integer_vars, integer_idx, plotter, bounds_tightening=False):
    # for each column in the tableau
    # construct a sparse vector for it
    # get the length of that vector via norm1 (plus 1 if we're an int column)
    # add our cut: sum_j(x_j/a_j)
    
    norm = 2
    current = integer_vars.X
    radius = np.linalg.norm(current - target, norm)
    if radius <= relaxed.params.FeasibilityTol:
        return False  # TODO: tolerance should apply to each component individually?
    
    if len(current) < 7:
        print("   Gap to target:", radius, ", Score:", relaxed.getObjective().getValue(), ",", current, "to", target)
    else:
        print("   Gap to target:", radius, ", Score:", relaxed.getObjective().getValue())
    if plotter is not None:
        plotter.add_ball(radius)

    variables = relaxed.getVars()  # TODO: pass this in as it's expensive?
    constraints = relaxed.getConstrs()  # wish we didn't have to use this one
    
    basis = gu.read_basis(relaxed)
    tableau, col_to_var, negated_rows = gu.read_tableau(relaxed, basis, extra_rows=1)
    negated_vars = [basis[nr] for nr in negated_rows]
    
    # drop the rows of non-integer variables:
    to_drop = [i for i, base in enumerate(basis) if base not in integer_idx]
    tableau = np.delete(tableau, to_drop, axis=0)  # TODO: don't even bother to read them in
    basis = np.delete(basis, to_drop)

    # Conforti has negative vectors with 1 at row=col, with the rest negated.
    # However, empirically, it seems that the opposite is what we really want (gurobi-specific or ineq. standardization issue)
    int_cols = [i for i, c in enumerate(col_to_var) if c in integer_idx]
    tableau[-1, int_cols] = -1  # use our extra row to store the col==row -> 1
    lengths = np.linalg.norm(tableau, norm, axis=0)
    lengths /= radius

    if bounds_tightening:  # assuming this is only true on a feasible target
        tol = relaxed.params.IntFeasTol
        for i, base in enumerate(basis):
            all_same_sign = np.all(tableau[i, :] > -tol) or np.all(tableau[i, :] < tol)  # TODO: find a more efficient call for this
            if all_same_sign and target[integer_idx[base]] > current[integer_idx[base]]:
                new_con = relaxed.addConstr(variables[base] >= target[integer_idx[base]])
                print("  Tightened", variables[base].VarName, ">=", target[integer_idx[base]])
                if plotter is not None:
                    relaxed.update()
                    plotter.add_constraint(new_con, color='xkcd:blue green')
            elif all_same_sign and target[integer_idx[base]] < current[integer_idx[base]]:
                new_con = relaxed.addConstr(-variables[base] >= -target[integer_idx[base]])
                print("  Tightened", variables[base].VarName, "<=", target[integer_idx[base]])
                if plotter is not None:
                    relaxed.update()
                    plotter.add_constraint(new_con, color='xkcd:blue green')
    
    relaxed.update()
    def find_variable(index):
        if index < len(variables):
            # handle inverted variables (SCIP and Gurobi both have this silliness)
            try:
                if variables[index].VBasis == -2:  # not yet sure this is correct
                    return variables[index].X - variables[index]
                assert variables[index].VBasis == -1  # not handling -3 yet
            except:
                pass  # TODO: figure out why they don't always have a VBasis & X attribute at this point
            return variables[index]
        # if only gurobi gave us access to their slack variables...
        # instead, we have to solve for it:
        cons_idx = index - len(variables)
        constraint = constraints[cons_idx]
        lhs, sense, rhs = relaxed.getRow(constraint), constraint.Sense, constraint.RHS
        if sense == '<':
             return rhs - lhs
        elif sense == '>':
            return lhs - rhs
        else:
            return 0.0  # is this right for equality constraints?
    
    summed_terms = gp.quicksum(lengths[i] * find_variable(j) for i, j in enumerate(col_to_var))
    new_con = relaxed.addConstr(summed_terms >= 1)

    print("   Moved:", 1.0 / np.linalg.norm(lengths, norm))
    def find_variable_value(index):
        if index < len(variables):
            try:
                if variables[index].VBasis == -2:
                    return 0.0  # not yet sure what to do here
                return variables[index].X
            except:
                return 0.0
        cons_idx = index - len(variables)
        constraint = constraints[cons_idx]
        try:
            return constraint.Slack
        except:
            return 0.0
            
    full_current = sum(lengths[i] * find_variable_value(j) for i, j in enumerate(col_to_var))
    assert full_current < 1.0    
    # what does it mean to take a relaxed point in full space and compare only its want-to-be-integer components to another?
    if plotter is not None:
        relaxed.update()
        plotter.add_constraint(new_con)
    return True

def run_cuts_to_relaxed_sol(instances, cut_function, bounds_tightening=True, loops=8, graph_2D_3D=True):
    for instance in instances:
        model: gp.Model = instance.as_gurobi_model()
        print("Running model:", model.ModelName)
        model.params.LogToConsole = 0
        model.update()
        plotter = pu.create(model) if graph_2D_3D else None

        exact_model = model.copy()
        exact_model.update()
        exact_vars = gp.MVar.fromlist([v for v in exact_model.getVars() if v.VType != 'C'])
        z = exact_model.addMVar(exact_vars.shape, lb=-gp.GRB.INFINITY)
        exact_model.setObjective(z.sum(), gp.GRB.MINIMIZE)
        
        int_vars, int_idx = gu.relax_int_or_bin_to_continuous(model)
        if bounds_tightening:
            gu.standardize_lt_to_gt(model)
            cons1 = gu.standardize_ub_to_constr(model)
            cons2 = gu.standardize_lb_to_constr(model)
            model.update()
            if plotter is not None:
                for con in cons1 + cons2:
                    plotter.add_constraint(con, color='xkcd:steel blue')

        for i in range(loops):
            model.optimize()
            if model.Status != gp.GRB.OPTIMAL:
                print("   FAILED! Status:", model.Status)
                break

            relaxed_x = int_vars.X
            ca = exact_model.addConstr(z >= exact_vars - relaxed_x)
            cb = exact_model.addConstr(z >= relaxed_x - exact_vars)
            exact_model.optimize()
            assert exact_model.Status == gp.GRB.OPTIMAL

            if not cut_function(model, exact_vars.X, int_vars, int_idx, plotter, bounds_tightening):
                print("   Final score of relaxed:", model.getObjective().getValue())
                break

            exact_model.remove(ca)
            exact_model.remove(cb)
        print("   Known best score:", instance.score if instance.known_optimum else "unknown")    
        if plotter is not None:
            plotter.render()

# test the cuts on simple examples:
run_cuts_to_relaxed_sol(list(el.get_instances().values()), add_balas_ball_cut, bounds_tightening=True)

In [ ]:
miplib_instances = ml.get_instances()
miplib_subset = [miplib_instances['air05'], miplib_instances['markshare2']]  # mas76
for instance in miplib_subset:
    mi = instance.as_gurobi_model()
    mi.params.Heuristics = 0
    mi.params.NoRelHeurTime = 2.0
    mi.params.DisplayInterval = 1
    mi.optimize()
    print("Expected optimum:", instance.score)

In [ ]:
import jsplib_loader as jl
jsplib_instances = jl.get_instances()
jsplib_subset = [jsplib_instances['abz7']]
for instance in jsplib_subset:
    mi = instance.as_gurobi_model()
    mi.params.Heuristics = 0
    mi.params.NoRelHeurTime = 100.0
    mi.params.DisplayInterval = 1
    mi.params.Cuts = 0
    mi.params.CutPasses = 0
    mi.optimize()
    print("Expected optimum:", instance.score)

In [ ]:
import importlib
importlib.reload(gu)

miplib_instances = ml.get_instances()
miplib_subset = [miplib_instances['air05'], miplib_instances['markshare2']]  # mas76
run_cuts_to_relaxed_sol(miplib_subset[0:1], add_balas_ball_cut)

In [19]:
import jsplib_loader as jl
jsplib_instances = jl.get_instances()
jsplib_subset = [jsplib_instances['abz4']]
run_cuts_to_relaxed_sol(jsplib_subset, add_balas_ball_cut, bounds_tightening=True, loops=10)
# jsplib_subset[0].as_gurobi_model().optimize()

Set parameter AggFill to value 10
Set parameter GomoryPasses to value 1
Running model: abz4
   Relaxed 216 variables on abz4
   Negated 0 constraints on abz4
   Standardized 216 upper bounds to be constraints
   Standardized 0 lower bounds to be constraints
   Gap to target: 0.6537663199482285 , Score: 505.0
   Moved: 0.05120692153656265
   Gap to target: 0.7397892688467739 , Score: 505.0
   Moved: 0.04107231513022534
   Gap to target: 0.6737838211976667 , Score: 505.0
   Moved: 0.03747493118290158
   Gap to target: 0.6738283750953945 , Score: 505.0
   Moved: 0.020304901835319994
   Gap to target: 0.7084523477333385 , Score: 505.0
   Moved: 0.019462851270765934
   Gap to target: 0.6540311739027478 , Score: 505.0
   Moved: 0.009131549202392066
   Gap to target: 0.863751502746742 , Score: 505.0
   Moved: 0.012132145925883298
   Gap to target: 0.8302223913426394 , Score: 505.0
   Moved: 0.04611848930681454
   Gap to target: 0.6942234077467743 , Score: 505.0
   Moved: 0.03720948441773238
 

In [10]:
# run comparisons:
def exit_early(model: gp.Model, where):
    if where == gp.GRB.Callback.MIPNODE:
        if model.cbGet(gp.GRB.Callback.MIPNODE_NODCNT) > 0:
            dualBound = model.cbGet(gp.GRB.Callback.MIPNODE_OBJBND)
            primalBound = model.cbGet(gp.GRB.Callback.MIPNODE_OBJBST)
            print("   Gurobi's best dual bound:", dualBound, ", best primal:", primalBound)
            model.terminate()

def compare_cuts(instances):
    for instance in instances:
        mi = instance.as_gurobi_model()
        if mi.NumVars * mi.NumConstrs * 8 > (1<<32):
            print("Skipping {mi.ModelName} because it is too big for us")
            continue

        print()
        print("Attempting Gurobi's aggressive cuts:")
        mi.params.Cuts = 3
        mi.params.CutPasses = 10
        mi.params.LogToConsole = 0
        mi.optimize(exit_early)

        print()
        print("Attempting Gurobi's NoRel:")
        del mi
        mi = instance.as_gurobi_model()
        mi.params.Heuristics = 0
        mi.params.NoRelHeurTime = 8.0
        mi.params.Cuts = 0
        mi.params.CutPasses = 0
        mi.params.LogToConsole = 0
        mi.optimize(exit_early)
        
        run_cuts_to_relaxed_sol([instance], add_balas_ball_cut, loops=12, bounds_tightening=False, graph_2D_3D=False)
        print("Expected optimum:", instance.score)
        print()
        print()

import jsplib_loader as jl
jsplib_instances = jl.get_instances()
jsplib_subset = [jsplib_instances['abz7']]

miplib_instances = ml.get_instances()
miplib_subset = [miplib_instances['air05'], miplib_instances['markshare2']]  # mas76

example_instances = el.get_instances()

compare_cuts(miplib_instances.values())

Set parameter Presolve to value 0
Set parameter Heuristics to value 0
Set parameter Cuts to value 0
Set parameter Presolve to value 0
Set parameter Heuristics to value 0
Set parameter Cuts to value 0
Set parameter Presolve to value 0
Set parameter Heuristics to value 0
Set parameter Cuts to value 0
Set parameter Presolve to value 0
Set parameter Heuristics to value 0
Set parameter Cuts to value 0
Set parameter Presolve to value 0
Set parameter Heuristics to value 0
Set parameter Cuts to value 0
Set parameter Presolve to value 0
Set parameter Heuristics to value 0
Set parameter Cuts to value 0
Set parameter Presolve to value 0
Set parameter Heuristics to value 0
Set parameter Cuts to value 0
Read MPS format model from file mip2017_benchmark\revised-submissions\miplib2010_50v-10\instances\50v-10.mps.gz
Reading time = 0.06 seconds
50v-10: 233 rows, 2013 columns, 2745 nonzeros
Running model: 50v-10
   Relaxed 1647 variables on 50v-10
   Gap to target: 3.551616983313252 , Score: 2879.065686